# 7. Full pipeline: labels, cross-validation, final models, ensemble, issue priority, analyses

| | |
|---|---|
| **Stage** | 7 of 9 |
| **Purpose** | The single notebook that produces the final results: it merges all label sources, cross-validates Tasks 2 and 3 against baselines, trains the final Task 3 model and a multi-seed ensemble, re-scores all sentence-aspect pairs, computes issue priority, runs the version and market analyses, the baselines and the error analysis, and exports every table with a run manifest. Each cross-validation fold and ensemble seed is saved, so an interrupted run resumes. |
| **Inputs** | `data/processed/{game}_clean.csv`<br>`data/annotation_v2/gold_set_combined.csv`<br>`outputs/sentiment_audit.csv`<br>`outputs/supplementary/* (S5, IAA and adjudication files)`<br>`outputs/predictions/{game}_pairs.parquet, {game}_aspects.parquet`<br>`outputs/dashboard/overall_{game}.parquet`<br>`models/task1_bert_combined`<br>`outputs/bert_summary_v2.json` |
| **Outputs** | `outputs/final_v4/ (tables, figures, results_registry.json, run_manifest.json, GameSense_results_v4.xlsx)`<br>`models/task3_v4`<br>`outputs/predictions_v4, outputs/predictions_v4_ens` |
| **Runtime** | GPU required (Colab T4). A full run resumes from saved folds and seeds; set `DRY_RUN = True` first for a short smoke test. |
| **Depends on** | Stages 1 to 6 |
| **Configuration** | Set `PROJECT_ROOT` in the first code cell if your Drive folder is not `MyDrive/GameSense`. |

> **Main notebook.** Every number in the report comes from this notebook. Earlier notebooks (03 to 05) produce the inputs it reads.

## 0. Environment and configuration

All switches are in the first code cell: `DRY_RUN` (smoke test), `CONTESTED_POLICY`, which cross-validations to run, the ensemble size and the paths. Only the folders listed in the header are written.

In [ ]:
# ===== 0. Setup & configuration =====
%pip install -q -U transformers accelerate openpyxl

import os, sys, json, gc, time, math, hashlib, platform, warnings, datetime
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass
import torch, torch.nn as nn

# ---------------- user switches ----------------
PROJECT_ROOT = "/content/drive/MyDrive/GameSense"
DRY_RUN = False               # True = tiny smoke test (1 epoch, 2 folds, small data); outputs go to *_dryrun folders
CONTESTED_POLICY = "exclude"  # S5 items where annotators disagree and nobody adjudicated yet: "exclude" or "a1" (keep annotator 1)
RUN_TASK3_CV = True           # 5-fold grouped CV for Task 3 (BERT + TF-IDF baseline)
RUN_TASK2_CV = True           # 5-fold grouped CV for Task 2 (BERT + TF-IDF baseline)
RUN_ENSEMBLE = True           # multi-seed Task 3 ensemble (used for the priority tables)
N_ENS_SEEDS = 5
FORCE_RETRAIN_FINAL = False   # True = retrain the final Task 3 model even if it already exists
DEPLOY_THR2 = 0.30            # deployed Task 2 threshold (as in deploy_config.json)
N_BOOT = 1000                 # bootstrap replicates for confidence intervals
VERSION_TRANSITIONS = [("royalmatch", "37281.0"), ("royalmatch", "37743.0"), ("genshin", "6.7.0")]  # (game, app version)
VERSION_WINDOW_DAYS = 14

MODEL_NAME = "bert-base-uncased"
SEED, EPOCHS, BS, LR, MAX_LEN, N_SPLITS = 42, 6, 16, 2e-5, 96, 5     # Task 3 recipe, identical to v2/v3
T2_EPOCHS, T2_BS, T2_LR = 20, 16, 5e-5                                # Task 2 CV recipe, identical to v1 CV
if DRY_RUN:
    EPOCHS, N_SPLITS, T2_EPOCHS, N_ENS_SEEDS, N_BOOT = 1, 2, 1, 2, 50

USE_CUDA = torch.cuda.is_available()
assert USE_CUDA or DRY_RUN, "No GPU: Runtime -> Change runtime type -> T4 GPU (CPU is only allowed for DRY_RUN)."
device = torch.device("cuda" if USE_CUDA else "cpu")

sys.path.insert(0, f"{PROJECT_ROOT}/scripts")
import importlib
if "aspect_taxonomy" in sys.modules:
    importlib.reload(sys.modules["aspect_taxonomy"])
from aspect_taxonomy import ASPECT_TAXONOMY, ASPECT_IDS      # never redefine the taxonomy inline
GAME_SLUGS = ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]
SENT_LABELS = ["negative", "neutral", "positive"]
L2I = {l: i for i, l in enumerate(SENT_LABELS)}
COMMON = ["performance_stability", "monetization_value", "advertisement", "level_difficulty_balance",
          "content_progression", "paywall", "account_support"]
GENRE = {"royalmatch": [], "genshin": ["gacha_rates", "character_operator_balance"],
         "mlbb": ["character_operator_balance", "matchmaking", "cheating"],
         "pubgm": ["cheating", "matchmaking"], "arknights": ["gacha_rates", "character_operator_balance", "grind_progression"]}
CAUTION = ["advertisement", "account_support", "cheating", "paywall"]   # aspects whose sentiment failed or is thin in earlier checks

# ---------------- input files (read-only) ----------------
SUPP = f"{PROJECT_ROOT}/outputs/supplementary"
GOLD_PATH = f"{PROJECT_ROOT}/data/annotation_v2/gold_set_combined.csv"
AUDIT_PATH = f"{PROJECT_ROOT}/outputs/sentiment_audit.csv"
S5_PATH = f"{SUPP}/s5_annotated.csv"
S5_KEY = f"{SUPP}/s5_key_DO_NOT_OPEN_BEFORE_ANNOTATING.csv"
S5_VERIFY = f"{SUPP}/s5_verify_annotated_by_annotator2.csv"
S5_ADJ = f"{SUPP}/s5_adjudicated.csv"
IAA_ADJ = f"{SUPP}/iaa_disagreements.csv"
IAA_FILES = [f"{SUPP}/iaa_annotator2.csv", f"{SUPP}/iaa_annotator2_topup.csv"]
GOLD_FIX = f"{SUPP}/gold_corrections.csv"
OLD_PRED = f"{PROJECT_ROOT}/outputs/predictions"          # v2 predictions (pairs + aspects), read-only

# ---------------- output folders (the only places this notebook writes) ----------------
SFX = "_dryrun" if DRY_RUN else ""
OUT = f"{PROJECT_ROOT}/outputs/final_v4{SFX}"
FIG = f"{OUT}/figures"
MODEL_DIR = f"{PROJECT_ROOT}/models/task3_v4{SFX}"
PRED_DIR = f"{PROJECT_ROOT}/outputs/predictions_v4{SFX}"
PRED_ENS = f"{PROJECT_ROOT}/outputs/predictions_v4_ens{SFX}"
for d in (OUT, FIG, MODEL_DIR, PRED_DIR, PRED_ENS):
    os.makedirs(d, exist_ok=True)
def safe_out(p):
    p = os.path.abspath(p)
    ok = [os.path.abspath(d) for d in (OUT, MODEL_DIR, PRED_DIR, PRED_ENS)]
    assert any(p == d or p.startswith(d + os.sep) for d in ok), f"refusing to write outside the v4 folders: {p}"
    return p
def write_supp(df, name):                         # the one file allowed outside the v4 folders (adjudication template)
    assert name == "s5_adjudicated.csv", name
    df.to_csv(f"{SUPP}/{name}", index=False)

# ---------------- results registry + table store (everything reported later comes from here) ----------------
REG_PATH = f"{OUT}/results_registry.json"
REG = json.load(open(REG_PATH)) if os.path.exists(REG_PATH) else {}
TABLES = {}
def _js(o):
    if hasattr(o, "item"): return o.item()
    if isinstance(o, (np.ndarray,)): return o.tolist()
    return str(o)
def reg(section, **kw):
    REG.setdefault(section, {}).update(kw)
    json.dump(REG, open(safe_out(REG_PATH), "w"), indent=2, default=_js)
def save_table(df, name, index=False):
    df.to_csv(safe_out(f"{OUT}/{name}.csv"), index=index)
    TABLES[name] = df.reset_index() if index else df
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50)

print(f"device={device} | DRY_RUN={DRY_RUN} | torch={torch.__version__} | out={OUT}")

## 1. Preflight: are all input files present?

Lists every input with its status. The run stops if a required file is missing.

In [ ]:
# ===== 1. Preflight: which input files exist and how complete are they =====
def n_rows(p, col=None, valid=None):
    if not os.path.exists(p):
        return "MISSING", ""
    d = pd.read_csv(p)
    if col is None:
        return "ok", f"{len(d)} rows"
    lab = d[col].fillna("").astype(str).str.strip().str.lower() if col in d.columns else pd.Series([""] * len(d))
    return "ok", f"{len(d)} rows, {int(lab.isin(valid).sum())} labeled"

checks = [("gold (annotator 1)", GOLD_PATH, None, None), ("audit (annotator 1)", AUDIT_PATH, "human_label", SENT_LABELS + ["na"]),
          ("S5 annotated (annotator 1)", S5_PATH, "human_label", SENT_LABELS + ["na"]), ("S5 key", S5_KEY, None, None),
          ("S5 verification (annotator 2)", S5_VERIFY, "annotator2_label", SENT_LABELS + ["na"]),
          ("S5 adjudication", S5_ADJ, "final_label", SENT_LABELS + ["na"]),
          ("IAA disagreements (adjudicated)", IAA_ADJ, "adjudicated_label", SENT_LABELS + ["none"]),
          ("gold corrections (optional)", GOLD_FIX, "new_label", SENT_LABELS + ["drop"])]
checks += [(f"IAA annotator 2 file {i + 1}", p, None, None) for i, p in enumerate(IAA_FILES)]
checks += [(f"{s}: clean csv", f"{PROJECT_ROOT}/data/processed/{s}_clean.csv", None, None) for s in GAME_SLUGS]
checks += [(f"{s}: v2 pairs", f"{OLD_PRED}/{s}_pairs.parquet", None, None) for s in GAME_SLUGS]
checks += [(f"{s}: v2 aspects", f"{OLD_PRED}/{s}_aspects.parquet", None, None) for s in GAME_SLUGS]
required = {GOLD_PATH, AUDIT_PATH, S5_PATH, S5_KEY} | {f"{PROJECT_ROOT}/data/processed/{s}_clean.csv" for s in GAME_SLUGS} \
           | {f"{OLD_PRED}/{s}_pairs.parquet" for s in GAME_SLUGS}
rows = []
for name, p, col, valid in checks:
    if p.endswith(".parquet"):
        st, extra = ("ok", "") if os.path.exists(p) else ("MISSING", "")
    else:
        st, extra = n_rows(p, col, valid)
    rows.append({"input": name, "status": st, "detail": extra, "required": "yes" if p in required else "optional"})
pf = pd.DataFrame(rows)
print(pf.to_string(index=False))
missing_required = pf[(pf["status"] == "MISSING") & (pf["required"] == "yes")]
assert missing_required.empty, f"required inputs missing: {missing_required['input'].tolist()}"
save_table(pf, "00_preflight")
print("\nAll required inputs are present.")

## 2. Merge all label sources into the Task 3 training table

Sources: gold labels (annotator 1), the IAA adjudication, optional gold corrections, the human audit pairs and the S5 validation items (annotator 1, annotator 2 and adjudication). The merge logic is the same as in the earlier training notebook; every count is printed and saved so it can be quoted in the report.

In [ ]:
# ===== 2. Resolve every label source and build the Task 3 training table =====
def rid(uid):                                   # sentence_uid = reviewId + "_" + sentence index
    return str(uid).rsplit("_", 1)[0]
clean = lambda s: s.fillna("").astype(str).str.strip().str.lower()
pending = []

# ---------- A. gold pairs (annotator 1) ----------
g = pd.read_csv(GOLD_PATH)
rows = []
for _, r in g.iterrows():
    for a in ASPECT_IDS:
        if pd.to_numeric(r.get(f"gold_{a}"), errors="coerce") == 1:
            s = r.get(f"gold_{a}_sentiment")
            if isinstance(s, str) and s in L2I:
                rows.append({"sentence_uid": r["sentence_uid"], "reviewId": rid(r["sentence_uid"]), "game": r["game"],
                             "aspect": a, "sentence_text": r["sentence_text"], "label": s, "source": "gold", "label_status": "single"})
gold_pairs = pd.DataFrame(rows)
gold_pairs_orig = gold_pairs.copy()             # kept for the Task 2 label update later
n_gold_original = len(gold_pairs)

# ---------- B. IAA adjudication ----------
iaa = dict.fromkeys(["disagreements", "adjudicated", "unadjudicated", "relabeled", "dropped", "added", "kept_same", "invalid"], 0)
if os.path.exists(IAA_ADJ):
    dis = pd.read_csv(IAA_ADJ)
    dis["adj"] = clean(dis["adjudicated_label"])
    iaa["disagreements"] = len(dis)
    added = []
    for _, d in dis.iterrows():
        if d["adj"] == "":
            iaa["unadjudicated"] += 1; continue
        if d["adj"] not in SENT_LABELS + ["none"]:
            iaa["invalid"] += 1; continue
        iaa["adjudicated"] += 1
        m = (gold_pairs["sentence_uid"] == d["sentence_uid"]) & (gold_pairs["aspect"] == d["aspect"])
        if m.any():
            if d["adj"] == "none":
                gold_pairs = gold_pairs[~m]; iaa["dropped"] += 1
            elif gold_pairs.loc[m, "label"].iloc[0] != d["adj"]:
                gold_pairs.loc[m, ["label", "label_status"]] = [d["adj"], "iaa_adjudicated"]; iaa["relabeled"] += 1
            else:
                iaa["kept_same"] += 1
        elif d["adj"] in SENT_LABELS:
            added.append({"sentence_uid": d["sentence_uid"], "reviewId": rid(d["sentence_uid"]), "game": d["game"], "aspect": d["aspect"],
                          "sentence_text": d["sentence_text"], "label": d["adj"], "source": "gold", "label_status": "iaa_added"})
            iaa["added"] += 1
    gold_pairs = pd.concat([gold_pairs, pd.DataFrame(added)], ignore_index=True) if added else gold_pairs.reset_index(drop=True)
    if iaa["unadjudicated"]:
        pending.append(f"IAA: {iaa['unadjudicated']} of {iaa['disagreements']} disagreements have no adjudicated_label (gold label kept)")
else:
    pending.append("IAA: iaa_disagreements.csv not found")
print("IAA adjudication:", iaa)

# ---------- C. optional gold corrections ----------
if os.path.exists(GOLD_FIX):
    fix = pd.read_csv(GOLD_FIX)
    n_changed = n_dropped = 0; unmatched = []
    for _, f in fix.iterrows():
        m = (gold_pairs["sentence_uid"] == f["sentence_uid"]) & (gold_pairs["aspect"] == f["aspect"])
        if not m.any():
            unmatched.append((f["sentence_uid"], f["aspect"])); continue
        if f["new_label"] == "drop":
            gold_pairs = gold_pairs[~m]; n_dropped += 1
        elif f["new_label"] in L2I:
            gold_pairs.loc[m, ["label", "label_status"]] = [f["new_label"], "s9c_fix"]; n_changed += 1
    gold_pairs = gold_pairs.reset_index(drop=True)
    print(f"gold corrections: {n_changed} relabeled, {n_dropped} dropped, unmatched: {unmatched or 'none'}")

# ---------- D. audit pairs ----------
aud = pd.read_csv(AUDIT_PATH)
aud["human_label"] = clean(aud["human_label"])
aud = aud[aud["human_label"].isin(SENT_LABELS)]
audit_pairs = aud.rename(columns={"human_label": "label"})[["sentence_uid", "game", "aspect", "sentence_text", "label"]].copy()
audit_pairs["reviewId"] = audit_pairs["sentence_uid"].map(rid)
audit_pairs["source"], audit_pairs["label_status"] = "audit", "single"

# ---------- E. S5 items ----------
s5 = pd.read_csv(S5_PATH)
key = pd.read_csv(S5_KEY)
s5["a1"] = clean(s5["human_label"])
s5 = s5.merge(key[["item_id", "sentence_uid", "reviewId", "game", "stratum", "stars", "model_sentiment"]], on="item_id", how="left")
s5["a2"] = ""
n_verify_expected = n_verify_done = 0
if os.path.exists(S5_VERIFY):
    v = pd.read_csv(S5_VERIFY)
    n_verify_expected = len(v)
    v["a2"] = clean(v["annotator2_label"])
    v = v[v["a2"].isin(SENT_LABELS + ["na"])]
    n_verify_done = len(v)
    s5["a2"] = s5["item_id"].map(v.set_index("item_id")["a2"]).fillna("")
    if n_verify_done < n_verify_expected:
        pending.append(f"S5 verification: annotator 2 labeled {n_verify_done} of {n_verify_expected}")
else:
    pending.append("S5 verification file not found: S5 labels come from annotator 1 only")
adj5 = {}
if os.path.exists(S5_ADJ):
    a = pd.read_csv(S5_ADJ)
    a["f"] = clean(a["final_label"])
    adj5 = a[a["f"].isin(SENT_LABELS + ["na"])].set_index("item_id")["f"].to_dict()
def resolve(r):
    if r["item_id"] in adj5:
        return adj5[r["item_id"]], "adjudicated"
    if r["a2"] == "":
        return r["a1"], "single"
    if r["a1"] == r["a2"]:
        return r["a1"], "agreed"
    return (r["a1"] if CONTESTED_POLICY == "a1" else None), "contested"
rs = s5.apply(resolve, axis=1, result_type="expand")
s5["label_final"], s5["label_status"] = rs[0], rs[1]
contested = s5[s5["label_status"] == "contested"]
if len(contested):
    tmpl = pd.read_csv(S5_ADJ) if os.path.exists(S5_ADJ) else pd.DataFrame(columns=["item_id", "aspect", "sentence_text", "annotator1", "annotator2", "final_label", "note"])
    new = contested[~contested["item_id"].isin(tmpl["item_id"])][["item_id", "aspect", "sentence_text", "a1", "a2"]].rename(columns={"a1": "annotator1", "a2": "annotator2"})
    if len(new):
        new["final_label"], new["note"] = "", ""
        write_supp(pd.concat([tmpl, new], ignore_index=True), "s5_adjudicated.csv")
    pending.append(f"S5: {len(contested)} contested items have no final_label -> fill outputs/supplementary/s5_adjudicated.csv (currently '{CONTESTED_POLICY}')")
s5_pairs = s5[s5["label_final"].isin(SENT_LABELS)].rename(columns={"label_final": "label", "model_sentiment": "v2_pred"})[
    ["item_id", "sentence_uid", "reviewId", "game", "aspect", "sentence_text", "label", "stratum", "stars", "v2_pred", "label_status"]].copy()
s5_pairs["source"] = "s5"
print(f"S5: {len(s5)} items -> {len(s5_pairs)} used | status: {s5['label_status'].value_counts().to_dict()}")

# ---------- F. combine ----------
data = pd.concat([gold_pairs, audit_pairs, s5_pairs], ignore_index=True)
dup = data.duplicated(["sentence_uid", "aspect"], keep="first")
if dup.any():
    print(f"WARNING: dropping {int(dup.sum())} duplicate (sentence, aspect) rows that appear in more than one source")
    data = data[~dup].reset_index(drop=True)
data["y"] = data["label"].map(L2I)
try:
    mk = pd.concat([pd.read_csv(f"{PROJECT_ROOT}/data/processed/{s}_clean.csv", usecols=["reviewId", "market"], dtype=str) for s in GAME_SLUGS])
    data["market"] = data["reviewId"].map(mk.drop_duplicates("reviewId").set_index("reviewId")["market"]).fillna("unknown")
except Exception as e:
    data["market"] = "unknown"; print("market column unavailable:", e)

STATUS = "FINAL" if not pending else "PRELIMINARY"
print(f"\n===== LABEL RESOLUTION REPORT | status: {STATUS} =====")
print(f"gold pairs: {n_gold_original} originally -> {int((data['source'] == 'gold').sum())} after IAA/corrections | audit: {int((data['source'] == 'audit').sum())} | S5: {int((data['source'] == 's5').sum())}")
print("\nlabel_status by source:\n", data.groupby(["source", "label_status"]).size().unstack(fill_value=0))
for p in pending:
    print(" - PENDING:", p)
reg("labels", status=STATUS, pending=pending, iaa_adjudication=iaa, pairs_by_source=data["source"].value_counts().to_dict(),
    s5_status=s5["label_status"].value_counts().to_dict(), contested_policy=CONTESTED_POLICY, n_gold_pairs_original=n_gold_original)
data.to_csv(safe_out(f"{OUT}/training_table.csv"), index=False)
if DRY_RUN:
    data = pd.concat([data[data["y"] == c].sample(min(int((data["y"] == c).sum()), 40), random_state=0) for c in range(3)]).reset_index(drop=True)
print("\nTotal pairs used:", len(data), "| distinct reviews:", data["reviewId"].nunique())

## 3. Label quality overview (for the data section of the report)

Three views: the label distribution per aspect (which aspects have too few examples), the contribution of each label source, and the agreement with the second annotator (IAA).

In [ ]:
# ===== 3. Label quality overview =====
lab_tab = data.groupby(["aspect", "label"]).size().unstack(fill_value=0).reindex(columns=SENT_LABELS, fill_value=0)
lab_tab["total"] = lab_tab.sum(axis=1)
lab_tab["minority"] = lab_tab[SENT_LABELS].apply(lambda r: int(min(r["negative"], r["positive"])), axis=1)
lab_tab["flag"] = np.select([lab_tab["total"] < 30, lab_tab["minority"] < 5], ["n<30", "<5 in minority polarity"], default="")
lab_tab = lab_tab.reindex([a for a in ASPECT_IDS if a in lab_tab.index])
print("=== Task 3 training pairs by aspect x label ===")
print(lab_tab.to_string())
save_table(lab_tab, "01_label_counts_by_aspect", index=True)

src_tab = data.groupby(["source", "label"]).size().unstack(fill_value=0).reindex(columns=SENT_LABELS, fill_value=0)
game_tab = data.groupby(["game", "label"]).size().unstack(fill_value=0).reindex(columns=SENT_LABELS, fill_value=0)
print("\n=== by source ===\n", src_tab.to_string(), "\n\n=== by game ===\n", game_tab.to_string())
save_table(src_tab, "01_label_counts_by_source", index=True); save_table(game_tab, "01_label_counts_by_game", index=True)

# ---- IAA (second annotator) ----
iaa_json = f"{SUPP}/iaa_summary.json"
if os.path.exists(iaa_json):
    iaa_sum = json.load(open(iaa_json))
    print("\n=== IAA summary (from S3b) ===")
    print(json.dumps(iaa_sum, indent=2)[:2500])
    reg("iaa", summary=iaa_sum)
else:
    print("\niaa_summary.json not found (run Checking.ipynb S3b first)")
iaa_pa = f"{SUPP}/iaa_per_aspect.csv"
if os.path.exists(iaa_pa):
    ipa = pd.read_csv(iaa_pa)
    print("\n=== IAA per aspect ===\n", ipa.to_string(index=False))
    save_table(ipa, "01_iaa_per_aspect")

## 4. Task 3 model code (sentiment of a sentence-aspect pair, 3 classes)

Same recipe as the earlier versions: `bert-base-uncased`, input `[CLS] sentence [SEP] aspect name`, cross-entropy with square-root-smoothed class weights, 6 epochs, learning rate 2e-5.

In [ ]:
# ===== 4. Task 3 model code (same recipe as v2/v3) =====
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, f1_score, classification_report, confusion_matrix, accuracy_score, average_precision_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding, set_seed)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)
ASPECT_LABEL = {a: ASPECT_TAXONOMY[a]["label"] for a in ASPECT_IDS}

class PairDS(Dataset):
    def __init__(self, df):
        self.enc = tokenizer(list(df["sentence_text"]), [ASPECT_LABEL[a] for a in df["aspect"]], truncation=True, max_length=MAX_LEN)
        self.y = df["y"].tolist()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item

class WTrainer(Trainer):
    def __init__(self, cw=None, **kw):
        super().__init__(**kw)
        self.cw = cw
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=self.cw.to(out.logits.device))(out.logits, labels)
        return (loss, out) if return_outputs else loss

def soft_weights(y):
    cnt = np.bincount(y, minlength=3).astype(float)
    cw = len(y) / (3 * np.maximum(cnt, 1))
    return torch.tensor(np.clip(np.sqrt(cw), 0.5, 3.0), dtype=torch.float)

def train_args(out_dir, n_rows, seed=SEED, **kw):          # seed also controls data order, so ensemble members really differ
    steps = math.ceil(n_rows / BS) * EPOCHS
    return TrainingArguments(output_dir=out_dir, num_train_epochs=EPOCHS, per_device_train_batch_size=BS,
                             per_device_eval_batch_size=64, learning_rate=LR, warmup_steps=int(0.1 * steps), weight_decay=0.01,
                             eval_strategy="no", save_strategy="no", fp16=USE_CUDA, report_to="none", seed=seed, **kw)

def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def fit_pair_model(df, seed, tmp="/content/t3v4_tmp", **kw):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3,
                                                              id2label={i: l for i, l in enumerate(SENT_LABELS)}, label2id=L2I)
    tr = WTrainer(cw=soft_weights(df["y"].values), model=model, args=train_args(tmp, len(df), seed=seed, **kw),
                  train_dataset=PairDS(df), data_collator=collator)
    tr.train()
    return tr, model

def train_predict(train_df, test_df, seed):
    tr, model = fit_pair_model(train_df, seed, logging_strategy="no", disable_tqdm=True)
    logits = tr.predict(PairDS(test_df)).predictions
    del tr, model
    gc.collect()
    if USE_CUDA:
        torch.cuda.empty_cache()
    return softmax(logits)

@torch.inference_mode()
def predict_logits(texts, tok, model, batch_size=128, max_length=MAX_LEN, pair_texts=None):
    order = np.argsort([len(t) for t in texts])
    out = None
    for start in range(0, len(texts), batch_size):
        idx = order[start:start + batch_size]
        batch = [texts[i] for i in idx]
        enc = (tok(batch, truncation=True, max_length=max_length, padding=True, return_tensors="pt") if pair_texts is None else
               tok(batch, [pair_texts[i] for i in idx], truncation=True, max_length=max_length, padding=True, return_tensors="pt"))
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_CUDA):
            logits = model(**enc).logits
        logits = logits.float().cpu().numpy()
        if out is None:
            out = np.zeros((len(texts), logits.shape[1]), dtype=np.float32)
        out[idx] = logits
    return out
print("Task 3 model code ready")

## 5. Task 3 · 5-fold cross-validation (BERT and a TF-IDF baseline on identical folds)

- Folds are grouped by **review** (sentences of one review never appear in both training and test) and stratified by sentiment class.
- Every fold is saved as soon as it finishes; after an interruption the run resumes from the saved folds.
- The TF-IDF + logistic regression baseline uses exactly the same folds so the comparison is fair.

In [ ]:
# ===== 5. Task 3: grouped 5-fold CV, BERT + TF-IDF baseline on identical folds =====
data_hash = hashlib.sha256("|".join(data["sentence_uid"].astype(str) + data["aspect"] + data["label"] + data["source"]).encode()).hexdigest()[:16]
hp = f"{OUT}/data_hash.txt"
if os.path.exists(hp) and open(hp).read().strip() != data_hash:
    for f in os.listdir(OUT):
        if f.startswith("t3_oof_fold"):
            os.remove(f"{OUT}/{f}")
    print("training table changed since the last run: old Task 3 fold files removed")
open(safe_out(hp), "w").write(data_hash)

splits = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(data, data["y"], groups=data["reviewId"].values))
oof = np.zeros((len(data), 3)); have_cv = False
if RUN_TASK3_CV:
    for k, (tr_i, te_i) in enumerate(splits, 1):
        fp = f"{OUT}/t3_oof_fold{k}.npy"
        if os.path.exists(fp):
            oof[te_i] = np.load(fp); print(f"fold {k}/{N_SPLITS}: loaded from a previous run"); continue
        t0 = time.time()
        oof[te_i] = train_predict(data.iloc[tr_i], data.iloc[te_i], SEED + k)
        np.save(safe_out(fp), oof[te_i])
        print(f"fold {k}/{N_SPLITS}  train={len(tr_i)}  test={len(te_i)}  {(time.time() - t0) / 60:.1f} min")
    have_cv = True
    np.save(safe_out(f"{OUT}/t3_oof.npy"), oof)
elif os.path.exists(f"{OUT}/t3_oof.npy") and np.load(f"{OUT}/t3_oof.npy").shape[0] == len(data):
    oof = np.load(f"{OUT}/t3_oof.npy"); have_cv = True
    print("RUN_TASK3_CV=False: loaded the saved Task 3 out-of-fold predictions")
else:
    print("RUN_TASK3_CV=False and no saved OOF found: Task 3 evaluation will be skipped")

# ---- TF-IDF + logistic regression baseline on the same folds ----
def pair_text(d):
    return (d["aspect"].map(ASPECT_LABEL) + " || " + d["sentence_text"]).tolist()
base_oof = np.zeros((len(data), 3))
for tr_i, te_i in splits:
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    Xtr = vec.fit_transform(pair_text(data.iloc[tr_i]))
    clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, data["y"].iloc[tr_i])
    full = np.zeros((len(te_i), 3)); full[:, clf.classes_] = clf.predict_proba(vec.transform(pair_text(data.iloc[te_i])))
    base_oof[te_i] = full
print("TF-IDF baseline done")

## 6. Task 3 · Evaluation (overall, confidence intervals, per aspect and per source)

Three main measures are reported:
- **three-class macro-F1** (the proposal's measure; the neutral class has very few examples and pulls it down),
- **negative-vs-rest macro-F1** (closest to the practical use: is an aspect being complained about?),
- **negative-vs-positive macro-F1** (non-neutral pairs only).

Confidence intervals come from a **cluster bootstrap by review** (1,000 resamples); the BERT minus baseline difference also gets an interval.

In [ ]:
# ===== 6. Task 3 evaluation =====
if not have_cv:
    print("No Task 3 CV results available: skip this cell")
else:
    y = data["y"].values
    pred_b, pred_t = oof.argmax(1), base_oof.argmax(1)
    def stats(yy, pp):
        m = yy != 1
        return {"macro_f1_3class": f1_score(yy, pp, average="macro", labels=[0, 1, 2], zero_division=0),
                "macro_f1_neg_vs_rest": f1_score((yy == 0).astype(int), (pp == 0).astype(int), average="macro", zero_division=0),
                "macro_f1_neg_vs_pos": f1_score((yy[m] == 0).astype(int), (pp[m] == 0).astype(int), average="macro", zero_division=0) if m.sum() else np.nan,
                "accuracy": accuracy_score(yy, pp)}
    sb, st = stats(y, pred_b), stats(y, pred_t)

    gid = pd.factorize(data["reviewId"])[0]
    groups = [np.where(gid == g_)[0] for g_ in range(gid.max() + 1)]
    rng = np.random.default_rng(SEED); brow = []
    for _ in range(N_BOOT):
        pick = rng.integers(0, len(groups), len(groups))
        ix = np.concatenate([groups[g_] for g_ in pick])
        a_, b_ = stats(y[ix], pred_b[ix]), stats(y[ix], pred_t[ix])
        brow.append({**{f"b_{k}": v for k, v in a_.items()}, **{f"t_{k}": v for k, v in b_.items()}, **{f"d_{k}": a_[k] - b_[k] for k in a_}})
    bt = pd.DataFrame(brow); lo, hi = bt.quantile(0.025), bt.quantile(0.975)
    rows = []
    for k in sb:
        rows.append({"metric": k, "BERT": round(sb[k], 3), "BERT_95CI": f"[{lo['b_' + k]:.3f}, {hi['b_' + k]:.3f}]",
                     "TFIDF": round(st[k], 3), "TFIDF_95CI": f"[{lo['t_' + k]:.3f}, {hi['t_' + k]:.3f}]",
                     "diff(BERT-TFIDF)": round(sb[k] - st[k], 3), "diff_95CI": f"[{lo['d_' + k]:.3f}, {hi['d_' + k]:.3f}]"})
    t3_overall = pd.DataFrame(rows)
    print(f"=== Task 3 pooled out-of-fold results (n={len(data)} pairs, {data['reviewId'].nunique()} reviews) ===")
    print(t3_overall.to_string(index=False)); save_table(t3_overall, "02_task3_overall")

    fold_rows = [{"fold": k, "n": len(te), **{f"BERT_{a}": round(v, 3) for a, v in stats(y[te], pred_b[te]).items() if a in ("macro_f1_3class", "macro_f1_neg_vs_rest")}} for k, (_, te) in enumerate(splits, 1)]
    ft = pd.DataFrame(fold_rows); print("\nper-fold:\n", ft.to_string(index=False)); save_table(ft, "02_task3_per_fold")

    rep = pd.DataFrame(classification_report(y, pred_b, labels=[0, 1, 2], target_names=SENT_LABELS, output_dict=True, zero_division=0)).T.round(3)
    print("\n=== BERT per-class report ===\n", rep.to_string()); save_table(rep, "02_task3_per_class_bert", index=True)
    cm = confusion_matrix(y, pred_b, labels=[0, 1, 2])
    cmdf = pd.DataFrame(cm, index=[f"human_{l}" for l in SENT_LABELS], columns=[f"model_{l}" for l in SENT_LABELS])
    print("\nconfusion matrix (rows = human, cols = model):\n", cmdf.to_string()); save_table(cmdf, "02_task3_confusion", index=True)
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(4.2, 3.6)); ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(3)); ax.set_yticks(range(3)); ax.set_xticklabels(SENT_LABELS); ax.set_yticklabels(SENT_LABELS)
        ax.set_xlabel("model"); ax.set_ylabel("human"); ax.set_title("Task 3 BERT (out-of-fold)")
        for i in range(3):
            for j in range(3):
                ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
        fig.tight_layout(); fig.savefig(safe_out(f"{FIG}/task3_confusion.png"), dpi=160); plt.close(fig)
    except Exception as e:
        print("figure skipped:", e)

    # ---- per aspect ----
    data["pred"], data["p_neg"] = pred_b, oof[:, 0]; data["pred_base"] = pred_t
    rows = []
    for a in ASPECT_IDS:
        m = data[data["aspect"] == a]
        if not len(m):
            continue
        isneg = m["y"] == 0; npos = int((m["y"] == 2).sum())
        rows.append({"aspect": a, "n": len(m), "neg": int(isneg.sum()), "neu": int((m["y"] == 1).sum()), "pos": npos,
                     "acc_negVsRest_BERT": round(float(((m["pred"] == 0) == isneg).mean()), 2),
                     "acc_negVsRest_TFIDF": round(float(((m["pred_base"] == 0) == isneg).mean()), 2),
                     "acc_if_all_negative": round(float(isneg.mean()), 2),
                     "AUC_BERT": round(float(roc_auc_score(isneg.astype(int), m["p_neg"])), 2) if 0 < isneg.sum() < len(m) else np.nan,
                     "pos_recall_BERT": round(float(((m["pred"] == 2) & (m["y"] == 2)).sum() / npos), 2) if npos else np.nan})
    t3_aspect = pd.DataFrame(rows)
    t3_aspect["note"] = np.where(t3_aspect["n"] < 30, "n<30", "")
    t3_aspect.loc[(t3_aspect["pos"] < 5) & (t3_aspect["note"] == ""), "note"] = "<5 positives"
    print("\n=== per aspect (compare acc_negVsRest with acc_if_all_negative) ===\n", t3_aspect.to_string(index=False)); save_table(t3_aspect, "02_task3_by_aspect")

    # ---- per source / game ----
    def grp(col):
        r = []
        for k_, m in data.groupby(col):
            isneg = m["y"] == 0
            r.append({col: k_, "n": len(m), "human_neg_share": round(float(isneg.mean()), 2), "acc_negVsRest": round(float(((m["pred"] == 0) == isneg).mean()), 2),
                      "AUC": round(float(roc_auc_score(isneg.astype(int), m["p_neg"])), 2) if 0 < isneg.sum() < len(m) else np.nan})
        return pd.DataFrame(r)
    for col in ("source", "game"):
        tt = grp(col); print(f"\n=== by {col} ===\n", tt.to_string(index=False)); save_table(tt, f"02_task3_by_{col}")

    # ---- v2 vs v4 on S5 items (v2 never saw them; CV predictions are out-of-fold) ----
    s5d = data[data["source"] == "s5"].copy()
    if len(s5d) and "v2_pred" in s5d:
        s5d["v2_id"] = s5d["v2_pred"].map(L2I)
        def negprec(d, col, stratum):
            x = d[(d["stratum"] == stratum) & (d[col] == 0)]
            return f"{(x['y'] == 0).sum()}/{len(x)}" if len(x) else "-"
        r = []
        for a in sorted(s5d["aspect"].unique()):
            d = s5d[s5d["aspect"] == a]; hp_ = d[d["y"] == 2]
            r.append({"aspect": a, "n": len(d), "hi_negPrec_v2": negprec(d, "v2_id", "hi_star_random"), "hi_negPrec_v4": negprec(d, "pred", "hi_star_random"),
                      "lo_negPrec_v2": negprec(d, "v2_id", "lo_star_random"), "lo_negPrec_v4": negprec(d, "pred", "lo_star_random"),
                      "posFound_v2": f"{int((hp_['v2_id'] != 0).sum())}/{len(hp_)}", "posFound_v4": f"{int((hp_['pred'] != 0).sum())}/{len(hp_)}",
                      "acc_v2": round(float(((d["v2_id"] == 0) == (d["y"] == 0)).mean()), 2), "acc_v4": round(float(((d["pred"] == 0) == (d["y"] == 0)).mean()), 2)})
        s5c = pd.DataFrame(r)
        print("\n=== S5 items: v2 vs this run (hi/lo = 4-5 / 1-2 star items; negPrec = predicted-negative that humans also called negative) ===\n", s5c.to_string(index=False))
        save_table(s5c, "02_task3_s5_v2_vs_v4")
    reg("task3_cv", n_pairs=len(data), n_reviews=int(data["reviewId"].nunique()), bert=sb, tfidf=st,
        ci95={k: [float(lo[k]), float(hi[k])] for k in lo.index}, data_hash=data_hash)

## 7. Task 2 · Sentence-level aspect detection (5-fold cross-validation, multi-label)

- Labels are the **adjudicated gold**: aspects added by annotator 2 are included, aspects the adjudication rejected are removed.
- Folds are grouped by review and stratified by each sentence's rarest aspect; they are generated in this notebook, so the numbers differ slightly from earlier cross-validations.
- Models: multi-label BERT (BCE with square-root positive weights) and TF-IDF + logistic regression; both use the **game applicability mask** (probabilities of aspects that do not apply to the game are set to zero).
- Results are reported at the fixed deployed threshold (0.30). The threshold tuned on out-of-fold predictions is shown for reference only, because it is an optimistic estimate.

In [ ]:
# ===== 7a. Task 2: model code (multi-label BERT) =====
class MLDS(Dataset):
    def __init__(self, texts, Y):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.Y = Y.astype(np.float32).tolist()
    def __len__(self):
        return len(self.Y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.Y[i]
        return item

class MLTrainer(Trainer):
    def __init__(self, pw=None, **kw):
        super().__init__(**kw)
        self.pw = pw
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.BCEWithLogitsLoss(pos_weight=self.pw.to(out.logits.device))(out.logits, labels.float())
        return (loss, out) if return_outputs else loss

def bert_multilabel_fold(tr_texts, Ytr, te_texts, seed):
    set_seed(seed)
    pc = Ytr.sum(axis=0); nc = len(Ytr) - pc
    pw = torch.tensor(np.clip(np.sqrt(nc / np.maximum(pc, 1)), 1.0, 6.0), dtype=torch.float)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(ASPECT_IDS), problem_type="multi_label_classification",
                                                              id2label={i: a for i, a in enumerate(ASPECT_IDS)}, label2id={a: i for i, a in enumerate(ASPECT_IDS)})
    steps = math.ceil(len(Ytr) / T2_BS) * T2_EPOCHS
    args = TrainingArguments(output_dir="/content/t2v4_tmp", num_train_epochs=T2_EPOCHS, per_device_train_batch_size=T2_BS, per_device_eval_batch_size=64,
                             learning_rate=T2_LR, warmup_steps=int(0.1 * steps), weight_decay=0.01, eval_strategy="no", save_strategy="no",
                             fp16=USE_CUDA, report_to="none", seed=seed, logging_strategy="no", disable_tqdm=True)
    tr = MLTrainer(pw=pw, model=model, args=args, train_dataset=MLDS(tr_texts, Ytr), data_collator=collator)
    tr.train()
    logits = tr.predict(MLDS(te_texts, np.zeros((len(te_texts), len(ASPECT_IDS))))).predictions
    del tr, model
    gc.collect()
    if USE_CUDA:
        torch.cuda.empty_cache()
    return 1 / (1 + np.exp(-logits))
print("Task 2 model code ready")

In [ ]:
# ===== 7b. Task 2: build resolved gold labels, run CV, evaluate =====
g2 = pd.read_csv(GOLD_PATH)
Y0 = np.stack([pd.to_numeric(g2[f"gold_{a}"], errors="coerce").fillna(0).astype(int).values for a in ASPECT_IDS], axis=1)
Y2 = Y0.copy(); uid_idx = {u: i for i, u in enumerate(g2["sentence_uid"])}
orig_set = set(zip(gold_pairs_orig["sentence_uid"], gold_pairs_orig["aspect"])); res_set = set(zip(gold_pairs["sentence_uid"], gold_pairs["aspect"]))
removed, added2 = orig_set - res_set, res_set - orig_set
for u, a in removed:
    if u in uid_idx: Y2[uid_idx[u], ASPECT_IDS.index(a)] = 0
for u, a in added2:
    if u in uid_idx: Y2[uid_idx[u], ASPECT_IDS.index(a)] = 1
print(f"Task 2 labels: {len(g2)} sentences | positives {int(Y0.sum())} -> {int(Y2.sum())} after adjudication ({len(added2)} added, {len(removed)} removed)")
mask2 = np.array([[1.0 if (a in COMMON or a in GENRE[gm]) else 0.0 for a in ASPECT_IDS] for gm in g2["game"]])
t2_texts = g2["sentence_text"].fillna("").tolist(); t2_rid = g2["sentence_uid"].map(rid).values
freq = Y2.sum(axis=0)
strat = np.array([(min(np.where(r == 1)[0], key=lambda j: freq[j]) if r.sum() else -1) for r in Y2])
splits2 = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(t2_texts, strat, groups=t2_rid))

# ---- TF-IDF + one-vs-rest logistic regression baseline ----
P2_base = np.zeros(Y2.shape)
for tr_i, te_i in splits2:
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    Xtr = vec.fit_transform([t2_texts[i] for i in tr_i])
    clf = OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight="balanced")).fit(Xtr, Y2[tr_i])
    P2_base[te_i] = clf.predict_proba(vec.transform([t2_texts[i] for i in te_i]))

# ---- BERT (checkpointed per fold) ----
P2_bert = np.zeros(Y2.shape); have_t2 = False
if RUN_TASK2_CV:
    for k, (tr_i, te_i) in enumerate(splits2, 1):
        fp = f"{OUT}/t2_oof_fold{k}.npy"
        if os.path.exists(fp) and np.load(fp).shape[0] == len(te_i):
            P2_bert[te_i] = np.load(fp); print(f"fold {k}/{N_SPLITS}: loaded"); continue
        t0 = time.time()
        P2_bert[te_i] = bert_multilabel_fold([t2_texts[i] for i in tr_i], Y2[tr_i], [t2_texts[i] for i in te_i], SEED + k)
        np.save(safe_out(fp), P2_bert[te_i]); print(f"fold {k}/{N_SPLITS}  {(time.time() - t0) / 60:.1f} min")
    have_t2 = True

if have_t2:
    def macro_ap(Yt, P):
        return float(np.mean([average_precision_score(Yt[:, j], P[:, j]) for j in range(Yt.shape[1]) if Yt[:, j].sum() > 0]))
    def f1s(Yt, P, thr):
        pr = (P >= thr).astype(int)
        return f1_score(Yt, pr, average="macro", zero_division=0), f1_score(Yt, pr, average="micro", zero_division=0)
    Pb, Pt = P2_bert * mask2, P2_base * mask2
    print(f"\nMasked gold labels falling in not-applicable aspects: {int((Y2 * (1 - mask2)).sum())}")
    rows = []
    for name, P in (("TF-IDF + LR", Pt), ("BERT", Pb)):
        fold_ap = [macro_ap(Y2[te], P[te]) for _, te in splits2]
        best = max(np.arange(0.15, 0.61, 0.05), key=lambda t: f1s(Y2, P, t)[1])
        m_f, mi_f = f1s(Y2, P, DEPLOY_THR2); bm_f, bmi_f = f1s(Y2, P, best)
        rows.append({"model": name, "macro_AP": round(macro_ap(Y2, P), 3), "fold_AP_mean": round(np.mean(fold_ap), 3), "fold_AP_sd": round(np.std(fold_ap), 3),
                     f"macroF1@{DEPLOY_THR2:.2f}": round(m_f, 3), f"microF1@{DEPLOY_THR2:.2f}": round(mi_f, 3),
                     "best_thr(OOF-tuned)": round(float(best), 2), "macroF1@best": round(bm_f, 3), "microF1@best": round(bmi_f, 3)})
    t2_overall = pd.DataFrame(rows); print("\n=== Task 2 pooled out-of-fold results ===\n", t2_overall.to_string(index=False)); save_table(t2_overall, "03_task2_overall")
    diffs = [macro_ap(Y2[te], Pb[te]) - macro_ap(Y2[te], Pt[te]) for _, te in splits2]
    print("paired per-fold macro-AP difference (BERT - TF-IDF):", np.round(diffs, 3), "| BERT better in", int(sum(d > 0 for d in diffs)), f"/{len(diffs)} folds")

    r = []
    for j, a in enumerate(ASPECT_IDS):
        n_ = int(Y2[:, j].sum())
        r.append({"aspect": a, "support": n_, "AP_TFIDF": round(average_precision_score(Y2[:, j], Pt[:, j]), 3) if n_ else np.nan,
                  "AP_BERT": round(average_precision_score(Y2[:, j], Pb[:, j]), 3) if n_ else np.nan,
                  "F1_TFIDF": round(f1_score(Y2[:, j], (Pt[:, j] >= DEPLOY_THR2).astype(int), zero_division=0), 3),
                  "F1_BERT": round(f1_score(Y2[:, j], (Pb[:, j] >= DEPLOY_THR2).astype(int), zero_division=0), 3)})
    t2_aspect = pd.DataFrame(r); t2_aspect["note"] = np.where(t2_aspect["support"] < 15, "support<15", "")
    t2_aspect.loc[(t2_aspect["F1_BERT"] < 0.6) & (t2_aspect["note"] == ""), "note"] = "F1<0.60"
    print("\n=== Task 2 per aspect ===\n", t2_aspect.to_string(index=False)); save_table(t2_aspect, "03_task2_by_aspect")
    sweep = pd.DataFrame([{"threshold": round(float(t), 2), **{f"{n_}_{k_}": round(v, 3) for n_, P in (("BERT", Pb), ("TFIDF", Pt)) for k_, v in zip(("macroF1", "microF1"), f1s(Y2, P, t))}}
                          for t in np.arange(0.15, 0.61, 0.05)])
    print("\nthreshold sweep (pooled OOF):\n", sweep.to_string(index=False)); save_table(sweep, "03_task2_threshold_sweep")
    bm_, bmi_ = f1s(Y2, Pb, DEPLOY_THR2)
    reg("task2_cv", n_sentences=len(g2), positives_before=int(Y0.sum()), positives_after=int(Y2.sum()), bert_macro_f1=bm_, bert_micro_f1=bmi_,
        bert_macro_ap=macro_ap(Y2, Pb), tfidf_macro_f1=f1s(Y2, Pt, DEPLOY_THR2)[0], tfidf_macro_ap=macro_ap(Y2, Pt), threshold=DEPLOY_THR2)
else:
    print("RUN_TASK2_CV=False: Task 2 CV skipped")

## 8. Final Task 3 model (trained on all labelled pairs and saved)

Skipped if the model already exists, unless `FORCE_RETRAIN_FINAL=True`.

In [ ]:
# ===== 8. Final Task 3 model on all labelled pairs =====
if os.path.exists(f"{MODEL_DIR}/config.json") and not FORCE_RETRAIN_FINAL:
    print(f"final model already exists -> {MODEL_DIR} (skipped; set FORCE_RETRAIN_FINAL=True to retrain)")
else:
    t0 = time.time()
    tr, model = fit_pair_model(data, SEED, logging_steps=20)
    tr.save_model(safe_out(MODEL_DIR)); tokenizer.save_pretrained(safe_out(MODEL_DIR))
    del tr, model; gc.collect()
    if USE_CUDA: torch.cuda.empty_cache()
    print(f"saved -> {MODEL_DIR}  ({(time.time() - t0) / 60:.1f} min)")

## 9. Re-score all sentence-aspect pairs with the final model

Task 2 stays frozen, so the set of (sentence, aspect) pairs does not change; only the sentiment is updated. The cell also reports the change in negative share from v2 to v4 as a sanity check (for example, advertisement in 4-5 star reviews should no longer be mostly negative).

In [ ]:
# ===== 9. Re-score the existing (sentence, aspect) pairs with the final model =====
tok3 = AutoTokenizer.from_pretrained(MODEL_DIR)
m3 = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()
assert [m3.config.id2label[i] for i in range(3)] == SENT_LABELS, "label order mismatch"
shift_rows = []
for slug in GAME_SLUGS:
    old = pd.read_parquet(f"{OLD_PRED}/{slug}_pairs.parquet")
    if DRY_RUN:
        old = old.sample(min(300, len(old)), random_state=0).copy()
    t0 = time.time()
    probs = softmax(predict_logits(old["sentence_text"].tolist(), tok3, m3, pair_texts=[ASPECT_LABEL[a] for a in old["aspect"]]))
    new = old.copy()
    new["sentiment"] = np.array(SENT_LABELS)[probs.argmax(1)]
    for j, lab in enumerate(SENT_LABELS):
        new[f"p_{lab}"] = probs[:, j].astype(np.float32)
    new.to_parquet(safe_out(f"{PRED_DIR}/{slug}_pairs.parquet"), index=False)
    print(f"{slug:11s} {len(new):6d} pairs | {(time.time() - t0) / 60:.1f} min | sentiment changed for {100 * (new['sentiment'] != old['sentiment']).mean():.1f}%")
    shift_rows.append(pd.DataFrame({"game": slug, "aspect": old["aspect"].values, "stars": old["review_score"].values,
                                    "neg_v2": (old["sentiment"] == "negative").values, "neg_v4": (new["sentiment"] == "negative").values}))
del m3; gc.collect()
if USE_CUDA: torch.cuda.empty_cache()
sh = pd.concat(shift_rows, ignore_index=True)
shift_aspect = sh.groupby("aspect").agg(n=("neg_v2", "size"), neg_v2=("neg_v2", "mean"), neg_v4=("neg_v4", "mean")).round(3)
sh["band"] = pd.cut(pd.to_numeric(sh["stars"], errors="coerce"), [0, 2, 3, 5], labels=["1-2 stars", "3 stars", "4-5 stars"])
shift_band = sh[sh["aspect"].isin(CAUTION)].groupby(["aspect", "band"], observed=True).agg(n=("neg_v2", "size"), neg_v2=("neg_v2", "mean"), neg_v4=("neg_v4", "mean")).round(3)
print("\n=== share predicted negative per aspect: v2 vs v4 ===\n", shift_aspect.to_string())
print("\n=== caution aspects by star band ===\n", shift_band.to_string())
save_table(shift_aspect, "04_shift_by_aspect", index=True); save_table(shift_band, "04_shift_caution_by_star_band", index=True)
reg("rescoring", pairs_rescored=int(len(sh)), neg_share_v4_overall=float(sh["neg_v4"].mean()), neg_share_v2_overall=float(sh["neg_v2"].mean()))

## 10. Multi-seed ensemble (used for issue priority)

The population-level negative share of a single model moves with the random seed. One model per seed is trained on all labelled pairs (the models are not saved, only the probabilities). The mean probability gives the final sentiment, and the run records:
- `p_neg_std`: the sentence-level spread across seeds (used to keep only stable examples),
- the negative rate of every (game, aspect) under every seed (used to add run-to-run variation to the ranking).

Each seed is saved when finished, so the run can be resumed. With `RUN_ENSEMBLE=False` the single-model predictions from section 9 are used instead.

In [ ]:
# ===== 10. Multi-seed ensemble =====
PRED_FOLDER = PRED_DIR
ens_neg_by_seed = None
if RUN_ENSEMBLE:
    ENS_SEEDS = [1000 + i for i in range(N_ENS_SEEDS)]
    P = {s: pd.read_parquet(f"{OLD_PRED}/{s}_pairs.parquet") for s in GAME_SLUGS}
    if DRY_RUN:
        P = {s: d.sample(min(300, len(d)), random_state=0).reset_index(drop=True) for s, d in P.items()}
    allp = pd.concat([d.assign(_slug=s) for s, d in P.items()], ignore_index=True)
    texts = allp["sentence_text"].tolist(); pair_texts = [ASPECT_LABEL[a] for a in allp["aspect"]]
    seed_probs = []
    for sd in ENS_SEEDS:
        fp = f"{PRED_ENS}/ens_probs_seed{sd}.npy"
        if os.path.exists(fp) and np.load(fp).shape[0] == len(allp):
            seed_probs.append(np.load(fp)); print(f"seed {sd}: loaded"); continue
        t0 = time.time()
        tr, model = fit_pair_model(data, sd, logging_strategy="no", disable_tqdm=True)
        model = model.to(device).eval()
        pr = softmax(predict_logits(texts, tokenizer, model, pair_texts=pair_texts)).astype(np.float32)
        np.save(safe_out(fp), pr); seed_probs.append(pr)
        del tr, model; gc.collect()
        if USE_CUDA: torch.cuda.empty_cache()
        print(f"seed {sd}: {(time.time() - t0) / 60:.1f} min")
    S = np.stack(seed_probs)                               # (seeds, pairs, 3)
    mean_p, sd_neg = S.mean(0), S[:, :, 0].std(0)
    for slug in GAME_SLUGS:
        m = (allp["_slug"] == slug).values
        out = P[slug].copy()
        out["sentiment"] = np.array(SENT_LABELS)[mean_p[m].argmax(1)]
        for j, lab in enumerate(SENT_LABELS):
            out[f"p_{lab}"] = mean_p[m][:, j].astype(np.float32)
        out["p_neg_std"] = sd_neg[m].astype(np.float32)
        out.to_parquet(safe_out(f"{PRED_ENS}/{slug}_pairs.parquet"), index=False)
    ens_neg_by_seed = pd.DataFrame({"game": allp["_slug"], "aspect": allp["aspect"],
                                    **{f"seed{sd}": (S[i].argmax(1) == 0) for i, sd in enumerate(ENS_SEEDS)}}).groupby(["game", "aspect"]).mean()
    save_table(ens_neg_by_seed.round(4), "05_ensemble_neg_rate_by_seed", index=True)
    print("\nrun-to-run spread of the negative rate (SD across seeds) by aspect, averaged over games:")
    print(ens_neg_by_seed.std(axis=1).groupby("aspect").mean().round(4).to_string())
    PRED_FOLDER = PRED_ENS
    reg("ensemble", seeds=ENS_SEEDS, mean_neg_share=float((mean_p.argmax(1) == 0).mean()), mean_seed_sd_neg_rate=float(ens_neg_by_seed.std(axis=1).mean()))
print("predictions used downstream:", PRED_FOLDER)

## 11. Issue priority (the core output of the proposal)

**Priority = negative mentions per 100 sentences** (mention rate times negative share, equivalent to normalised frequency times negative rate).

- Ranked per game; cross-game comparison uses only the 7 **common aspects**.
- Robustness 1: is the ranking stable when only 1-2 star reviews are used (Spearman)?
- Robustness 2: best and worst rank across the ensemble seeds.
- Reliability flags: too few training pairs, low AUC, known risky aspects and weak Task 2 detection are marked; the dashboard and the report should show them.

In [ ]:
# ===== 11. Issue priority =====
import pyarrow.parquet as pq
from scipy.stats import spearmanr
pairs = {s: pd.read_parquet(f"{PRED_FOLDER}/{s}_pairs.parquet") for s in GAME_SLUGS}

def sentence_counts(slug):
    p = f"{OLD_PRED}/{slug}_aspects.parquet"
    if not os.path.exists(p):
        return None, None
    names = pq.ParquetFile(p).schema.names
    total = pq.ParquetFile(p).metadata.num_rows
    band = None
    if "review_score" in names:
        sc = pd.to_numeric(pd.read_parquet(p, columns=["review_score"])["review_score"], errors="coerce")
        band = {"1-2": int((sc <= 2).sum()), "all": total}
    return total, band

rel_notes = {}
if "t3_aspect" in globals():
    for _, r in t3_aspect.iterrows():
        n_ = []
        if r["n"] < 30: n_.append("few Task 3 pairs")
        if r["pos"] < 5: n_.append("<5 positives")
        if pd.notna(r["AUC_BERT"]) and r["AUC_BERT"] < 0.75: n_.append("low AUC")
        rel_notes.setdefault(r["aspect"], []).extend(n_)
if "t2_aspect" in globals():
    for _, r in t2_aspect.iterrows():
        if r["support"] < 15 or r["F1_BERT"] < 0.60: rel_notes.setdefault(r["aspect"], []).append("weak Task 2 detection")
for a in CAUTION:
    rel_notes.setdefault(a, []).append("sentiment flagged in S5 check")

prio_rows = []
for s in GAME_SLUGS:
    d = pairs[s].copy(); d["is_neg"] = d["sentiment"] == "negative"
    total, band = sentence_counts(s)
    if total is None or DRY_RUN:
        total = max(len(d), 1)
        if not DRY_RUN: print(f"WARNING: {s}_aspects.parquet missing: sentence count falls back to the pair count (priority is NOT normalised correctly)")
    ag = d.groupby("aspect").agg(mentions=("is_neg", "size"), neg=("is_neg", "sum"))
    ag["game"] = s; ag["n_sentences"] = total
    ag["mentions_per100"] = 100 * ag["mentions"] / total
    ag["neg_share"] = ag["neg"] / ag["mentions"]
    ag["priority_neg_per100"] = 100 * ag["neg"] / total
    if band is not None and "review_score" in d.columns:
        sc = pd.to_numeric(d["review_score"], errors="coerce")
        ag["priority_1to2"] = 100 * d[sc <= 2].groupby("aspect")["is_neg"].sum().reindex(ag.index).fillna(0) / max(band["1-2"], 1)
    if ens_neg_by_seed is not None and s in ens_neg_by_seed.index.get_level_values(0):
        sub = ens_neg_by_seed.loc[s].reindex(ag.index)
        pri_seed = sub.mul(ag["mentions_per100"], axis=0)
        ag["neg_share_sd_seeds"] = sub.std(axis=1)
        ranks = pri_seed.rank(ascending=False, method="min")
        ag["rank_min_seeds"], ag["rank_max_seeds"] = ranks.min(axis=1), ranks.max(axis=1)
    prio_rows.append(ag.reset_index())
prio = pd.concat(prio_rows, ignore_index=True)
prio["rank"] = prio.groupby("game")["priority_neg_per100"].rank(ascending=False, method="min")
prio["reliability_notes"] = prio["aspect"].map(lambda a: "; ".join(dict.fromkeys(rel_notes.get(a, []))))
prio["reliability"] = np.where(prio["reliability_notes"] == "", "ok", "check")
prio = prio.sort_values(["game", "rank"]).reset_index(drop=True)
for c_ in ("mentions_per100", "neg_share", "priority_neg_per100", "priority_1to2", "neg_share_sd_seeds"):
    if c_ in prio: prio[c_] = prio[c_].round(3)
save_table(prio, "06_issue_priority_by_game")
for s in GAME_SLUGS:
    cols = [c_ for c_ in ("aspect", "mentions", "mentions_per100", "neg_share", "priority_neg_per100", "rank", "rank_min_seeds", "rank_max_seeds", "reliability") if c_ in prio]
    print(f"\n===== {s} (sentences: {int(prio[prio.game == s]['n_sentences'].iloc[0])}) =====")
    print(prio[prio.game == s][cols].to_string(index=False))

# ---- robustness 1: ranking under 1-2 star reviews only ----
if "priority_1to2" in prio:
    rr = []
    for s in GAME_SLUGS:
        d = prio[prio.game == s].dropna(subset=["priority_1to2"])
        if len(d) >= 3:
            rho = spearmanr(d["priority_neg_per100"], d["priority_1to2"])[0]
            top_all = set(d.nlargest(3, "priority_neg_per100")["aspect"]); top_12 = set(d.nlargest(3, "priority_1to2")["aspect"])
            rr.append({"game": s, "spearman_all_vs_1to2": round(float(rho), 2), "top3_overlap": f"{len(top_all & top_12)}/3"})
    rob = pd.DataFrame(rr); print("\n=== ranking robustness: all reviews vs 1-2 star only ===\n", rob.to_string(index=False)); save_table(rob, "06_priority_robustness_1to2")

# ---- cross-game comparison on COMMON aspects only ----
cg = prio[prio["aspect"].isin(COMMON)].pivot(index="aspect", columns="game", values="priority_neg_per100").reindex(COMMON)
print("\n=== cross-game comparison, common aspects only: negative mentions per 100 sentences ===\n", cg.round(2).to_string())
save_table(cg.round(3), "06_cross_game_common_priority", index=True)
cs = prio[prio["aspect"].isin(COMMON)].pivot(index="aspect", columns="game", values="neg_share").reindex(COMMON)
save_table(cs.round(3), "06_cross_game_common_neg_share", index=True)

try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, len(GAME_SLUGS), figsize=(4.2 * len(GAME_SLUGS), 4.2), sharex=False)
    for ax, s in zip(np.atleast_1d(axes), GAME_SLUGS):
        d = prio[prio.game == s].sort_values("priority_neg_per100")
        err = (d["neg_share_sd_seeds"] * d["mentions_per100"]).values if "neg_share_sd_seeds" in d else None
        ax.barh(d["aspect"], d["priority_neg_per100"], xerr=err, color=np.where(d["reliability"] == "ok", "#2E5597", "#C9A227"))
        ax.set_title(s, fontsize=10); ax.tick_params(labelsize=7); ax.set_xlabel("neg. mentions / 100 sentences", fontsize=8)
    fig.suptitle("Issue priority (gold = reliability check flagged)", fontsize=11); fig.tight_layout(); fig.savefig(safe_out(f"{FIG}/issue_priority.png"), dpi=150); plt.close(fig)
except Exception as e:
    print("figure skipped:", e)
reg("priority", top_issue_by_game={s: prio[(prio.game == s) & (prio["rank"] == 1)]["aspect"].tolist() for s in GAME_SLUGS})

## 12. Version analysis (descriptive only, no causal claims)

For each chosen (game, app version), the 5th percentile of that version's review dates is taken as the first-seen date, and the 14 days before and after are compared. Reported:
- the number of reviews in each window (the proposal requires at least 100) and the share on the new version,
- the change in average stars with a 95% bootstrap interval,
- the change in the share of reviews with a negative mention per aspect, with intervals (only aspects with at least `MIN_MENT` mentions in both windows).

When an interval includes 0, do not say that the update improved or worsened anything.

In [ ]:
# ===== 12. Version-aware pre/post analysis (descriptive only) =====
def parse_robust(s):
    try:
        t = pd.to_datetime(s, format="mixed", utc=True, errors="coerce")
    except (ValueError, TypeError):
        from dateutil import parser
        t = pd.to_datetime(s.map(lambda x: parser.parse(x) if isinstance(x, str) and x.strip() else pd.NaT), errors="coerce", utc=True)
    return t.dt.tz_convert(None)

B_V, MIN_N, MIN_MENT = (60 if DRY_RUN else 2000), 100, 20
rng_v = np.random.default_rng(SEED)
def boot_delta(pre, post):
    ip = rng_v.integers(0, len(pre), (B_V, len(pre))); iq = rng_v.integers(0, len(post), (B_V, len(post)))
    d = post[iq].mean(axis=1) - pre[ip].mean(axis=1)
    return np.percentile(d, [2.5, 97.5], axis=0)

star_rows, asp_rows = [], []
for slug, ver in VERSION_TRANSITIONS:
    try:
        rv = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv", usecols=["reviewId", "at", "score", "reviewCreatedVersion"], dtype={"reviewCreatedVersion": str})
        rv["at"] = parse_robust(rv["at"]); rv["score"] = pd.to_numeric(rv["score"], errors="coerce")
        rv["reviewCreatedVersion"] = rv["reviewCreatedVersion"].astype("string").str.split("_").str[0]   # Genshin: "6.7.0_4548.._4576.." -> "6.7.0"
        rv = rv.dropna(subset=["at", "score"]).drop_duplicates("reviewId")
        on_ver = rv[rv["reviewCreatedVersion"] == ver]
        if len(on_ver) < MIN_N:
            print(f"{slug} {ver}: only {len(on_ver)} reviews on this version -> skipped"); continue
        fs = on_ver["at"].quantile(0.05); w = pd.Timedelta(days=VERSION_WINDOW_DAYS)
        pre = rv[(rv["at"] >= fs - w) & (rv["at"] < fs)]; post = rv[(rv["at"] >= fs) & (rv["at"] < fs + w)]
        usable = len(pre) >= MIN_N and len(post) >= MIN_N
        share_new = float((post["reviewCreatedVersion"] == ver).mean()) if len(post) else np.nan
        row = {"game": slug, "version": ver, "first_seen": fs.date(), "n_pre": len(pre), "n_post": len(post), "post_share_on_new_version": round(share_new, 2), "usable(>=100 each)": usable}
        if len(pre) and len(post):
            lo_, hi_ = boot_delta(pre["score"].values.astype(float), post["score"].values.astype(float))
            row.update({"stars_pre": round(pre["score"].mean(), 2), "stars_post": round(post["score"].mean(), 2),
                        "stars_delta": round(post["score"].mean() - pre["score"].mean(), 2), "stars_delta_95CI": f"[{lo_:.2f}, {hi_:.2f}]",
                        "CI_excludes_0": bool(lo_ > 0 or hi_ < 0)})
        star_rows.append(row)
        if not usable:
            continue
        d = pairs[slug].copy(); d["reviewId"] = d["sentence_uid"].map(rid)
        negm = d[d["sentiment"] == "negative"].groupby(["reviewId", "aspect"]).size().unstack(fill_value=0) > 0
        for a in negm.columns:
            mp = negm[a].reindex(pre["reviewId"]).fillna(False).values.astype(np.float32); mq = negm[a].reindex(post["reviewId"]).fillna(False).values.astype(np.float32)
            if mp.sum() < MIN_MENT or mq.sum() < MIN_MENT:
                continue
            lo_, hi_ = boot_delta(mp, mq)
            asp_rows.append({"game": slug, "version": ver, "aspect": a, "pre_per100_reviews": round(float(100 * mp.mean()), 2), "post_per100_reviews": round(float(100 * mq.mean()), 2),
                             "delta": round(float(100 * (mq.mean() - mp.mean())), 2), "delta_95CI": f"[{100 * lo_:.2f}, {100 * hi_:.2f}]", "CI_excludes_0": bool(lo_ > 0 or hi_ < 0),
                             "caution": "sentiment flagged" if a in CAUTION else ""})
    except Exception as e:
        print(f"{slug} {ver}: version analysis failed -> {type(e).__name__}: {e}")
ver_stars = pd.DataFrame(star_rows); ver_asp = pd.DataFrame(asp_rows)
print("=== window sizes and star-rating change ===\n", ver_stars.to_string(index=False) if len(ver_stars) else "(none)")
print("\n=== change in share of reviews with a negative mention, per aspect ===\n", ver_asp.to_string(index=False) if len(ver_asp) else "(none)")
if len(ver_stars): save_table(ver_stars, "07_version_windows_and_stars")
if len(ver_asp): save_table(ver_asp, "07_version_aspect_changes")
reg("version_analysis", usable_transitions=int(ver_stars["usable(>=100 each)"].sum()) if len(ver_stars) else 0, transitions_tried=len(VERSION_TRANSITIONS))

## 12b. Market composition audit and window-boundary sensitivity

**Why.** The cleaned data mixes several Google Play markets (us, gb, id), and the market is tied to the collection batch; non-US reviews sit mostly in the last weeks. If a version window contains non-US reviews, a change in stars may only reflect a different reviewer population.

**Three checks**
1. Reviews, date range and average stars per game and market.
2. The share of non-US reviews in each 14-day window.
3. Sensitivity of each version window: all markets vs US only, and the window anchor (exact timestamp, midnight of that day, shifted by one or two days).

If shifting the window by a day or two changes the result noticeably, report a range instead of a single number.

In [ ]:
# ===== 12b. Market composition audit + window-boundary sensitivity =====
frames = []
for slug in GAME_SLUGS:
    d = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv", usecols=["reviewId", "at", "score", "market", "reviewCreatedVersion"],
                    dtype={"at": str, "reviewCreatedVersion": str})
    d["at"] = parse_robust(d["at"]); d["score"] = pd.to_numeric(d["score"], errors="coerce"); d["game"] = slug
    d["ver"] = d["reviewCreatedVersion"].astype("string").str.split("_").str[0]
    frames.append(d.dropna(subset=["at", "score"]).drop_duplicates("reviewId"))
allr = pd.concat(frames, ignore_index=True)
allr["us"] = allr["market"].eq("us")

# ---- 1. per game x market ----
mk1 = allr.groupby(["game", "market"]).agg(n=("reviewId", "size"), first=("at", "min"), last=("at", "max"), avg_stars=("score", "mean")).reset_index()
mk1["first"], mk1["last"] = mk1["first"].dt.date, mk1["last"].dt.date
print("=== 1. reviews per game and market ===")
print(mk1[mk1["n"] >= 20].round(2).to_string(index=False)); save_table(mk1, "07b_market_composition")

# ---- 2. non-US share per 14-day window ----
rows = []
for slug in GAME_SLUGS:
    d = allr[allr["game"] == slug]
    if (~d["us"]).mean() < 0.01:
        continue
    w = (d["at"] - d["at"].min().normalize()).dt.days // VERSION_WINDOW_DAYS
    for k, g_ in d.groupby(w):
        rows.append({"game": slug, "window_start": (d["at"].min().normalize() + pd.Timedelta(days=int(k) * VERSION_WINDOW_DAYS)).date(),
                     "n": len(g_), "nonUS_%": round(100 * float((~g_["us"]).mean()), 1)})
nus = pd.DataFrame(rows)
print("\n=== 2. non-US share per window (games with <1% non-US are omitted) ===")
print(nus.to_string(index=False) if len(nus) else "(all games are essentially US-only)")
if len(nus): save_table(nus, "07b_nonUS_share_by_window")

# ---- 3. sensitivity of each version transition ----
sens = []
for slug, ver in VERSION_TRANSITIONS:
    d = allr[allr["game"] == slug]
    on_ver = d[d["ver"] == ver]
    if len(on_ver) < MIN_N:
        continue
    t_exact = on_ver["at"].quantile(0.05)
    variants = {"exact timestamp": t_exact, "midnight of that day (S7d/S9f)": t_exact.normalize()}
    for sh_ in (-2, -1, 1, 2):
        variants[f"exact {sh_:+d} day"] = t_exact + pd.Timedelta(days=sh_)
    for vname, t0 in variants.items():
        for sname, sel in (("all markets", d), ("US only", d[d["us"]])):
            a = sel[(sel["at"] >= t0 - pd.Timedelta(days=VERSION_WINDOW_DAYS)) & (sel["at"] < t0)]
            b = sel[(sel["at"] >= t0) & (sel["at"] < t0 + pd.Timedelta(days=VERSION_WINDOW_DAYS))]
            if len(a) < 30 or len(b) < 30:
                continue
            lo_, hi_ = boot_delta(a["score"].values.astype(float), b["score"].values.astype(float))
            sens.append({"transition": f"{slug} {ver}", "window_anchor": vname, "subset": sname, "n_pre": len(a), "n_post": len(b),
                         "nonUS_post_%": round(100 * float((~b["us"]).mean()), 1), "stars_pre": round(a["score"].mean(), 2), "stars_post": round(b["score"].mean(), 2),
                         "stars_delta": round(b["score"].mean() - a["score"].mean(), 2), "delta_95CI": f"[{lo_:.2f}, {hi_:.2f}]", "CI_excludes_0": bool(lo_ > 0 or hi_ < 0)})
sens = pd.DataFrame(sens)
print("\n=== 3. version windows: market subset and window-boundary sensitivity (stars) ===")
print(sens.to_string(index=False) if len(sens) else "(none)")
if len(sens):
    save_table(sens, "07b_version_boundary_sensitivity")
    rng_tab = sens[sens["subset"] == "all markets"].groupby("transition")["stars_delta"].agg(["min", "max"]).round(2)
    print("\nrange of the star change across boundary choices (all markets):\n", rng_tab.to_string())
    reg("version_sensitivity", star_delta_range={k: [float(v["min"]), float(v["max"])] for k, v in rng_tab.iterrows()})

## 12c. Rule/lexicon baselines and error analysis for Tasks 2 and 3

**Why.** The proposal lists keyword/lexicon baselines (Task 2) and lexicon/rule baselines (Task 3), and requires class-level results and error analysis when targets are not met.

**What this section does**
- **Baselines.** Task 2: keyword matching on the taxonomy with the game mask. Task 3: two rule baselines (the small project word list and the Hu & Liu opinion lexicon, both with negation handling), compared with predicting 'negative' for everything, TF-IDF and BERT. The lexicon baselines work at sentence level and do not know which aspect is meant.
- **Error analysis.** Performance by game, which language markers (contrast words, negation, question marks, emoji, ALL CAPS, very short or long sentences) go with more errors, the most confident wrong examples, aspect pairs that are confused in Task 2, and the relation between an aspect's number of examples and its F1.

The language markers are simple transparent rules used only to describe where errors cluster; they are not a diagnosis of the cause.

In [ ]:
# ===== 12c. Rule/lexicon baselines + error analysis for Task 2 and Task 3 =====
import re
from scipy.stats import spearmanr

FLAGS = {
    "contrast word (but/however/although)": lambda s: bool(re.search(r"\b(but|however|although|though|yet)\b", s, re.I)),
    "negation": lambda s: bool(re.search(r"\b(not|no|never|cannot|can't|cant|don't|dont|doesn't|doesnt|didn't|didnt|won't|wont|isn't|isnt)\b", s, re.I)),
    "question mark": lambda s: "?" in s,
    "emoji or non-ASCII symbol": lambda s: any(ord(ch) > 127 for ch in s),
    "ALL-CAPS word": lambda s: bool(re.search(r"\b[A-Z]{3,}\b", s)),
    "short (<=5 words)": lambda s: len(s.split()) <= 5,
    "long (>=30 words)": lambda s: len(s.split()) >= 30,
}
def flag_df(texts):
    return pd.DataFrame({k: [f(str(t)) for t in texts] for k, f in FLAGS.items()})
def flag_text(row):
    return "; ".join(k for k, v in row.items() if v)
def error_by_flag(F, err, min_n=10):
    rows = []
    for c_ in F.columns:
        m = F[c_].values.astype(bool); e = np.asarray(err).astype(float)
        rows.append({"marker": c_, "n_with_marker": int(m.sum()), "error_rate_with": round(float(e[m].mean()), 3) if m.sum() else np.nan,
                     "n_without": int((~m).sum()), "error_rate_without": round(float(e[~m].mean()), 3) if (~m).sum() else np.nan,
                     "note": "few examples" if m.sum() < min_n else ""})
    return pd.DataFrame(rows)

# ---------------- Task 3: rule / lexicon baselines on the same labelled pairs ----------------
def project_rule(sentence):                         # the small word lists used for weak labelling during annotation
    NEGATORS = {"not", "no", "never", "n't", "cant", "can't", "cannot", "doesnt", "doesn't", "wont", "won't"}
    POS_WORDS = {"good", "great", "love", "loved", "fun", "amazing", "awesome", "enjoy", "best", "smooth", "easy", "worth"}
    NEG_WORDS = {"bad", "terrible", "awful", "hate", "hated", "annoying", "frustrating", "worst", "broken", "unfair", "ripoff", "greedy", "boring", "expensive"}
    score, negate = 0, False
    for tok in re.findall(r"[a-z']+", sentence.lower()):
        if tok in NEGATORS:
            negate = True; continue
        if tok in POS_WORDS: score += -1 if negate else 1
        elif tok in NEG_WORDS: score += 1 if negate else -1
        negate = False
    return "positive" if score > 0 else "negative" if score < 0 else "neutral"

def make_hu_liu():                                  # Hu & Liu (2004) opinion lexicon, as cited in the proposal
    try:
        import nltk
        nltk.download("opinion_lexicon", quiet=True)
        from nltk.corpus import opinion_lexicon
        pos, neg = set(opinion_lexicon.positive()), set(opinion_lexicon.negative())
    except Exception as e:
        print("Hu & Liu lexicon unavailable:", type(e).__name__, e); return None
    NEGW = {"not", "no", "never", "cannot", "can't", "cant", "don't", "dont", "doesn't", "doesnt", "didn't", "didnt", "won't", "wont", "isn't", "isnt", "without"}
    def f(sentence):
        toks = re.findall(r"[a-z']+", sentence.lower()); score = 0
        for i, t in enumerate(toks):
            v = 1 if t in pos else -1 if t in neg else 0
            if v and any(x in NEGW for x in toks[max(0, i - 2):i]): v = -v
            score += v
        return "positive" if score > 0 else "negative" if score < 0 else "neutral"
    return f

def stats3(yy, pp):
    return {"macro_f1_3class": f1_score(yy, pp, average="macro", labels=[0, 1, 2], zero_division=0),
            "macro_f1_neg_vs_rest": f1_score((yy == 0).astype(int), (pp == 0).astype(int), average="macro", zero_division=0),
            "accuracy": accuracy_score(yy, pp)}
y3, txt3 = data["y"].values, data["sentence_text"].tolist()
cands = [("Predict 'negative' for everything", np.zeros(len(y3), dtype=int)),
         ("Rule: small project word list + negation", np.array([L2I[project_rule(t)] for t in txt3]))]
hl = make_hu_liu()
if hl is not None:
    cands.append(("Rule: Hu & Liu opinion lexicon + negation", np.array([L2I[hl(t)] for t in txt3])))
if have_cv:
    cands += [("TF-IDF + logistic regression (cross-validated)", base_oof.argmax(1)), ("BERT (cross-validated)", oof.argmax(1))]
b3 = pd.DataFrame([{"model": n, "n_pairs": len(y3), **{k: round(v, 3) for k, v in stats3(y3, p).items()}} for n, p in cands])
print("=== Task 3 baselines on the same labelled pairs (sentence-level lexicons do not know which aspect is meant) ===")
print(b3.to_string(index=False)); save_table(b3, "09_baselines_task3")

# ---------------- Task 3: error analysis ----------------
if have_cv:
    d3 = data.copy(); d3["pred"] = oof.argmax(1); d3["p_neg"] = oof[:, 0]
    F3 = flag_df(d3["sentence_text"]); d3["markers"] = F3.apply(flag_text, axis=1)
    e3a = error_by_flag(F3, d3["pred"].values != d3["y"].values).rename(columns={"error_rate_with": "err3_with", "error_rate_without": "err3_without"})
    e3b = error_by_flag(F3, (d3["pred"].values == 0) != (d3["y"].values == 0))[["marker", "error_rate_with", "error_rate_without"]].rename(
        columns={"error_rate_with": "negVsRest_err_with", "error_rate_without": "negVsRest_err_without"})
    e3 = e3a.merge(e3b, on="marker")
    print("\n=== Task 3: error rate with vs without each language marker (3-class and negative-vs-rest) ===\n", e3.to_string(index=False))
    save_table(e3, "09_task3_error_by_marker")
    cols = ["game", "aspect", "source", "sentence_text", "human_label", "model_label", "p_negative", "markers"]
    def pack(df, kind):
        o = pd.DataFrame({"error_type": kind, "game": df["game"], "aspect": df["aspect"], "source": df["source"], "sentence_text": df["sentence_text"],
                          "human_label": df["label"], "model_label": df["pred"].map(dict(enumerate(SENT_LABELS))), "p_negative": df["p_neg"].round(3), "markers": df["markers"]})
        return o
    missed = pack(d3[(d3["y"] == 0) & (d3["pred"] != 0)].sort_values("p_neg").head(15), "missed complaint (human: negative)")
    alarm = pack(d3[(d3["y"] != 0) & (d3["pred"] == 0)].sort_values("p_neg", ascending=False).head(15), "false alarm (human: not negative)")
    ex3 = pd.concat([missed, alarm], ignore_index=True)
    print("\nmost confidently wrong examples (first 8 of each type):")
    for kind, g_ in ex3.groupby("error_type"):
        print("\n--", kind)
        for _, r in g_.head(8).iterrows():
            print(f"   [{r['game']}/{r['aspect']}] p_neg={r['p_negative']:.2f} | {r['sentence_text'][:150]}")
    save_table(ex3, "09_task3_error_examples")
    reg("error_analysis_task3", n_errors_3class=int((d3["pred"] != d3["y"]).sum()), n_pairs=len(d3))

# ---------------- Task 2: keyword baseline + error analysis ----------------
def keyword_hits(sentence):
    s = str(sentence).lower()
    return {a for a, meta in ASPECT_TAXONOMY.items() if any(kw in s for kw in meta.get("keywords", []))}
K2 = np.array([[1.0 if a in keyword_hits(t) else 0.0 for a in ASPECT_IDS] for t in t2_texts]) * mask2
kw_macro = f1_score(Y2, K2.astype(int), average="macro", zero_division=0); kw_micro = f1_score(Y2, K2.astype(int), average="micro", zero_division=0)
rows2 = [{"model": "Keyword matching on the aspect taxonomy (with game mask)", "macro_f1": round(kw_macro, 3), "micro_f1": round(kw_micro, 3), "macro_AP": np.nan}]
if have_t2:
    pred2 = (Pb >= DEPLOY_THR2).astype(int)
    for n_, P_ in (("TF-IDF + logistic regression (cross-validated)", Pt), ("BERT (cross-validated)", Pb)):
        m_, mi_ = f1s(Y2, P_, DEPLOY_THR2)
        rows2.append({"model": n_, "macro_f1": round(m_, 3), "micro_f1": round(mi_, 3), "macro_AP": round(macro_ap(Y2, P_), 3)})
b2 = pd.DataFrame(rows2)
print(f"\n=== Task 2 baselines (same {len(Y2)} gold sentences, threshold {DEPLOY_THR2} for the models) ===\n", b2.to_string(index=False)); save_table(b2, "09_baselines_task2")

if have_t2:
    ta = t2_aspect.copy()
    ta["F1_keyword"] = [round(f1_score(Y2[:, j], K2[:, j].astype(int), zero_division=0), 3) for j in range(len(ASPECT_IDS))]
    print("\n=== Task 2 per aspect, all three models ===\n", ta.to_string(index=False)); save_table(ta, "09_task2_by_aspect_all_models")
    rho, pv = spearmanr(ta["support"], ta["F1_BERT"])
    print(f"\nSpearman between an aspect's number of gold examples and its BERT F1: {rho:.2f} (p={pv:.3f}, n={len(ta)} aspects)")
    reg("error_analysis_task2", spearman_support_vs_f1=float(rho), p_value=float(pv))

    rows = []
    for slug in GAME_SLUGS:
        ix = (g2["game"] == slug).values
        Yg, pg = Y2[ix], pred2[ix]; cols_ = [j for j in range(len(ASPECT_IDS)) if Yg[:, j].sum() > 0]
        rows.append({"game": slug, "n_sentences": int(ix.sum()), "n_gold_aspect_labels": int(Yg.sum()),
                     "macro_f1 (aspects with gold labels)": round(f1_score(Yg[:, cols_], pg[:, cols_], average="macro", zero_division=0), 3),
                     "micro_f1": round(f1_score(Yg, pg, average="micro", zero_division=0), 3)})
    g2t = pd.DataFrame(rows); print("\n=== Task 2 by game ===\n", g2t.to_string(index=False)); save_table(g2t, "09_task2_by_game")

    F2 = flag_df(t2_texts); mk2 = F2.apply(flag_text, axis=1)
    e2 = error_by_flag(F2, (pred2 != Y2).any(axis=1)).rename(columns={"error_rate_with": "sentence_error_with", "error_rate_without": "sentence_error_without"})
    print("\n=== Task 2: share of sentences with at least one wrong aspect, with vs without each marker ===\n", e2.to_string(index=False)); save_table(e2, "09_task2_error_by_marker")

    fn = [(Pb[i, j], i, j) for i, j in zip(*np.where((Y2 == 1) & (pred2 == 0)))]
    fp = [(Pb[i, j], i, j) for i, j in zip(*np.where((Y2 == 0) & (pred2 == 1)))]
    def pack2(items, kind, reverse):
        items = sorted(items, key=lambda x: x[0], reverse=reverse)[:20]
        return pd.DataFrame([{"error_type": kind, "game": g2["game"].iloc[i], "aspect": ASPECT_IDS[j], "p_model": round(float(p), 3), "sentence_text": t2_texts[i],
                              "gold_aspects": ", ".join(ASPECT_IDS[k] for k in np.where(Y2[i] == 1)[0]), "predicted_aspects": ", ".join(ASPECT_IDS[k] for k in np.where(pred2[i] == 1)[0]),
                              "markers": mk2.iloc[i]} for p, i, j in items])
    ex2 = pd.concat([pack2(fn, "missed aspect (most confidently missed)", False), pack2(fp, "false alarm (most confident)", True)], ignore_index=True)
    save_table(ex2, "09_task2_error_examples")
    print("\nTask 2 examples (first 6 of each type):")
    for kind, g_ in ex2.groupby("error_type"):
        print("\n--", kind)
        for _, r in g_.head(6).iterrows():
            print(f"   [{r['game']}] {r['aspect']} p={r['p_model']:.2f} | gold: {r['gold_aspects'] or '-'} | {r['sentence_text'][:130]}")

    pair_counts = {}
    for i in range(len(Y2)):
        missed_ = np.where((Y2[i] == 1) & (pred2[i] == 0))[0]; extra_ = np.where((Y2[i] == 0) & (pred2[i] == 1))[0]
        for a_ in missed_:
            for b_ in extra_:
                pair_counts[(ASPECT_IDS[a_], ASPECT_IDS[b_])] = pair_counts.get((ASPECT_IDS[a_], ASPECT_IDS[b_]), 0) + 1
    conf = pd.DataFrame([{"missed_aspect": a_, "predicted_instead": b_, "sentences": n_} for (a_, b_), n_ in sorted(pair_counts.items(), key=lambda kv: -kv[1])[:15]])
    print("\n=== In sentences with both a miss and a false alarm: which aspect was predicted instead of which was missed ===\n", conf.to_string(index=False) if len(conf) else "(none)")
    if len(conf): save_table(conf, "09_task2_confusable_aspects")

## 12d. Error analysis for Task 1 (overall review sentiment)

The Task 1 test set is rebuilt as in the training notebook (five games combined, stratified by the star-derived label, 70/15/15, random seed 42) and scored with the saved `models/task1_bert_combined`. **If the rebuilt test macro-F1 equals the recorded 0.664, the test set is identical**; the cell checks this automatically.

Outputs: per-class precision and recall, the confusion matrix, performance by game, the **predictions per star rating** (to examine the neutral class), the most confident wrong reviews, and error rates by language marker. Requires a GPU.

In [ ]:
# ===== 12d. Task 1 (overall review sentiment): error analysis on the rebuilt held-out test set =====
RUN_TASK1_ERRORS = True
T1_MODEL_DIR = f"{PROJECT_ROOT}/models/task1_bert_combined"
if not RUN_TASK1_ERRORS or DRY_RUN:
    print("Task 1 error analysis skipped (RUN_TASK1_ERRORS=False or DRY_RUN=True)")
elif not os.path.exists(f"{T1_MODEL_DIR}/config.json"):
    print("Task 1 model not found at", T1_MODEL_DIR, "-> skipped")
else:
    from sklearn.model_selection import train_test_split
    LAB1 = ["negative", "neutral", "positive"]; L2I1 = {l: i for i, l in enumerate(LAB1)}
    frames = []
    for slug in GAME_SLUGS:
        d_ = pd.read_csv(f"{PROJECT_ROOT}/data/processed/{slug}_clean.csv", usecols=["reviewId", "content_clean", "overall_sentiment_weak", "score"]); d_["game"] = slug; frames.append(d_)
    rv = pd.concat(frames, ignore_index=True).dropna(subset=["content_clean", "overall_sentiment_weak"])
    rv["label_id"] = rv["overall_sentiment_weak"].map(L2I1)
    _, temp = train_test_split(rv, test_size=0.30, random_state=42, stratify=rv["label_id"])
    _, test1 = train_test_split(temp, test_size=0.50, random_state=42, stratify=temp["label_id"])
    test1 = test1.reset_index(drop=True)
    tok1 = AutoTokenizer.from_pretrained(T1_MODEL_DIR)
    m1 = AutoModelForSequenceClassification.from_pretrained(T1_MODEL_DIR).to(device).eval()
    assert [m1.config.id2label[i] for i in range(3)] == LAB1, "Task 1 label order mismatch"
    pr1 = softmax(predict_logits(test1["content_clean"].tolist(), tok1, m1, batch_size=64, max_length=256))
    del m1; gc.collect()
    if USE_CUDA: torch.cuda.empty_cache()
    test1["pred"] = pr1.argmax(1); y1 = test1["label_id"].values
    macro1 = f1_score(y1, test1["pred"], average="macro", zero_division=0)
    rec = None
    try:
        rec = json.load(open(f"{PROJECT_ROOT}/outputs/bert_summary_v2.json")).get("task1_overall_sentiment", {}).get("test_macro_f1_bert")
    except Exception:
        pass
    print(f"rebuilt test set: n={len(test1)} | macro-F1 = {macro1:.3f}" + (f" | recorded value {rec}" if rec is not None else ""))
    if rec is not None and abs(macro1 - float(rec)) > 0.005:
        print("WARNING: differs from the recorded value -> the rebuilt test set may not be identical (the clean CSVs may have changed).")
    elif rec is not None:
        print("OK: matches the recorded test result, so this is the same held-out test set.")

    rep = pd.DataFrame(classification_report(y1, test1["pred"], labels=[0, 1, 2], target_names=LAB1, output_dict=True, zero_division=0)).T.round(3)
    print("\n=== Task 1 per-class report ===\n", rep.to_string()); save_table(rep, "09_task1_overall", index=True)
    cm1 = pd.DataFrame(confusion_matrix(y1, test1["pred"], labels=[0, 1, 2]), index=[f"weak_{l}" for l in LAB1], columns=[f"model_{l}" for l in LAB1])
    print("\nconfusion matrix (rows = star-derived label, cols = model):\n", cm1.to_string()); save_table(cm1, "09_task1_confusion", index=True)
    byg = pd.DataFrame([{"game": s_, "n": int((test1["game"] == s_).sum()),
                         "macro_f1": round(f1_score(y1[test1["game"] == s_], test1.loc[test1["game"] == s_, "pred"], average="macro", zero_division=0), 3),
                         "accuracy": round(accuracy_score(y1[test1["game"] == s_], test1.loc[test1["game"] == s_, "pred"]), 3)} for s_ in GAME_SLUGS])
    print("\n=== Task 1 by game ===\n", byg.to_string(index=False)); save_table(byg, "09_task1_by_game")
    test1["stars"] = pd.to_numeric(test1["score"], errors="coerce")
    bys = pd.DataFrame([{"stars": int(s_), "n": int(len(g_)), **{f"predicted_{l}_%": round(100 * float((g_["pred"] == i).mean()), 1) for i, l in enumerate(LAB1)}}
                        for s_, g_ in sorted(test1.dropna(subset=["stars"]).groupby("stars"))])
    print("\n=== What the model predicts for each star rating (3-star reviews carry the 'neutral' label) ===\n", bys.to_string(index=False)); save_table(bys, "09_task1_by_star")
    F1m = flag_df(test1["content_clean"]); test1["markers"] = F1m.apply(flag_text, axis=1)
    e1 = error_by_flag(F1m, test1["pred"].values != y1).rename(columns={"error_rate_with": "error_with", "error_rate_without": "error_without"})
    print("\n=== Task 1: error rate with vs without each marker ===\n", e1.to_string(index=False)); save_table(e1, "09_task1_error_by_marker")
    for i, l in enumerate(LAB1): test1[f"p_{l}"] = pr1[:, i]
    def pack1(df, kind, pcol):
        return pd.DataFrame({"error_type": kind, "game": df["game"], "stars": df["stars"], "weak_label": df["label_id"].map(dict(enumerate(LAB1))),
                             "model_label": df["pred"].map(dict(enumerate(LAB1))), "p_model": df[pcol].round(3),
                             "review_text": df["content_clean"].str.slice(0, 300), "markers": df["markers"]})
    ex1 = pd.concat([
        pack1(test1[(test1["label_id"] == 0) & (test1["pred"] == 2)].sort_values("p_positive", ascending=False).head(10), "labelled negative (1-2 stars), model says positive", "p_positive"),
        pack1(test1[(test1["label_id"] == 2) & (test1["pred"] == 0)].sort_values("p_negative", ascending=False).head(10), "labelled positive (4-5 stars), model says negative", "p_negative"),
        pack1(test1[(test1["label_id"] == 1) & (test1["pred"] != 1)].assign(_p=lambda x: x[["p_negative", "p_positive"]].max(axis=1)).sort_values("_p", ascending=False).head(10).assign(p_model=lambda x: x["_p"]).drop(columns="_p"),
              "labelled neutral (3 stars), model says positive/negative", "p_model")], ignore_index=True)
    save_table(ex1, "09_task1_error_examples")
    print("\nexamples (first 5 of each type):")
    for kind, g_ in ex1.groupby("error_type"):
        print("\n--", kind)
        for _, r in g_.head(5).iterrows():
            print(f"   [{r['game']} {r['stars']}*] p={r['p_model']:.2f} | {r['review_text'][:140]}")
    reg("error_analysis_task1", n_test=len(test1), macro_f1_rebuilt=float(macro1), recorded=rec)

## 13. Comparison with the proposal's success criteria, run manifest and export

Collects data volume, the three task metrics, the version analysis and the reproducibility material. **A missed target is not a failure by itself**: section 5.4 of the proposal states that the numeric targets are directional and asks for class-level results and error analysis, which the tables above provide.

In [ ]:
# ===== 13. Proposal check, run manifest, Excel export =====
n_reviews = pd.concat([pd.read_csv(f"{PROJECT_ROOT}/data/processed/{s}_clean.csv", usecols=["reviewId"], dtype=str) for s in GAME_SLUGS])["reviewId"].nunique()
t1 = {}
bs_path = f"{PROJECT_ROOT}/outputs/bert_summary_v2.json"
if os.path.exists(bs_path):
    t1 = json.load(open(bs_path)).get("task1_overall_sentiment", {})
def status(v, tgt):
    return "n/a" if v is None or (isinstance(v, float) and np.isnan(v)) else ("met" if v >= tgt else "below target")
r3 = REG.get("task3_cv", {}); r2 = REG.get("task2_cv", {})
rows = [
    {"criterion": "Usable data volume", "target": ">=12,000 unique reviews", "result": f"{n_reviews:,} reviews", "status": status(n_reviews, 12000)},
    {"criterion": "Manually annotated sentences", "target": "~1,000", "result": f"{len(g2)} gold sentences (+ {int((data['source'] != 'gold').sum())} audit/S5 pairs)", "status": "below plan"},
    {"criterion": "Task 1 overall sentiment macro-F1", "target": ">=0.80 and > best non-DNN baseline", "result": f"BERT {t1.get('test_macro_f1_bert', 'n/a')} vs TF-IDF {t1.get('test_macro_f1_tfidf', 'n/a')} (recorded run)",
     "status": status(t1.get("test_macro_f1_bert"), 0.80)},
    {"criterion": "Task 2 aspect detection macro-F1", "target": ">=0.70", "result": (f"BERT {r2['bert_macro_f1']:.3f} vs TF-IDF {r2['tfidf_macro_f1']:.3f} @thr {r2['threshold']} (CV)" if r2 else "not run"),
     "status": status(r2.get("bert_macro_f1"), 0.70) if r2 else "n/a"},
    {"criterion": "Task 3 aspect sentiment macro-F1 (3-class)", "target": ">=0.70", "result": (f"BERT {r3['bert']['macro_f1_3class']:.3f} vs TF-IDF {r3['tfidf']['macro_f1_3class']:.3f} (CV)" if r3 else "not run"),
     "status": status(r3["bert"]["macro_f1_3class"], 0.70) if r3 else "n/a"},
    {"criterion": "Task 3 negative-vs-rest macro-F1 (supplementary)", "target": "(report)", "result": (f"BERT {r3['bert']['macro_f1_neg_vs_rest']:.3f} vs TF-IDF {r3['tfidf']['macro_f1_neg_vs_rest']:.3f}" if r3 else "not run"), "status": "reported"},
    {"criterion": "Version analysis", "target": ">=1 game, 14-day windows, >=100 reviews each", "result": f"{REG.get('version_analysis', {}).get('usable_transitions', 0)} usable transition(s)",
     "status": "met" if REG.get("version_analysis", {}).get("usable_transitions", 0) >= 1 else "not met"},
    {"criterion": "Issue profiles + cross-game comparison", "target": "ranked per game; common aspects normalised", "result": "06_issue_priority_by_game, 06_cross_game_common_*", "status": "produced"},
    {"criterion": "Dashboard", "target": "Streamlit app", "result": "not part of this notebook", "status": "pending"},
    {"criterion": "Reproducibility", "target": "README + dependency file", "result": "run_manifest.json written; README / requirements.txt still to write", "status": "partly"},
]
chk = pd.DataFrame(rows); print(chk.to_string(index=False)); save_table(chk, "08_proposal_check")

# ---- run manifest (reproducibility) ----
import sklearn, transformers
manifest = {"timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "dry_run": DRY_RUN, "python": platform.python_version(), "torch": torch.__version__,
            "transformers": transformers.__version__, "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__, "model": MODEL_NAME,
            "seeds": {"main": SEED, "ensemble": [1000 + i for i in range(N_ENS_SEEDS)]}, "task3_recipe": {"epochs": EPOCHS, "batch": BS, "lr": LR, "max_len": MAX_LEN, "folds": N_SPLITS},
            "task2_recipe": {"epochs": T2_EPOCHS, "batch": T2_BS, "lr": T2_LR, "threshold": DEPLOY_THR2}, "training_table_hash": data_hash, "label_status": REG.get("labels", {}).get("status")}
json.dump(manifest, open(safe_out(f"{OUT}/run_manifest.json"), "w"), indent=2)

# ---- one Excel workbook with every saved table ----
xlsx = safe_out(f"{OUT}/GameSense_results_v4.xlsx")
with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
    for f in sorted(os.listdir(OUT)):
        if f.endswith(".csv") and f != "training_table.csv":
            pd.read_csv(f"{OUT}/{f}").to_excel(xw, sheet_name=f[:-4][:31], index=False)
print("\nExcel workbook:", xlsx)
print("Files in", OUT, ":\n ", "\n  ".join(sorted(os.listdir(OUT))))

## 14. Wrap-up checklist

- Use **File > Save a copy in Drive** to keep a version with outputs.
- Every number in the report can be traced to a table in `outputs/final_v4/`; the Excel summary is `GameSense_results_v4.xlsx` and the raw record is `results_registry.json`.
- Table prefixes: `02_*` Task 3 evaluation, `03_*` Task 2 evaluation, `04_*` change from v2 to v4, `05_*` seed variation, `06_*` issue priority and cross-game comparison, `07_*` version analysis, `08_*` comparison with the proposal, `09_*` baselines and error analysis.
- If the label-resolution status is `PRELIMINARY`, some disagreements are still unadjudicated: fill them in `s5_adjudicated.csv` and rerun from section 2.
- The Streamlit dashboard reads `outputs/predictions_v4_ens/` and `06_issue_priority_by_game.csv` through notebook 09.